# Chapter 48: 3D Vision & Scene Reconstruction
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Mempelajari prinsip geometri multi-pandang (Epipolar Geometry), formulasi matriks proyeksi kamera pinhole, algoritma Direct Linear Transformation (DLT) berbasis SVD untuk triangulasi koordinat 3D, serta evaluasi eror reproyeksi spasial.

### 1. Landasan Geometri & Formulasi Triangulasi DLT
Model kamera proyektif memetakan titik 3D $\mathbf{X} = [X, Y, Z, 1]^T$ menjadi koordinat piksel homogen $\lambda \mathbf{x} = \mathbf{P} \mathbf{X}$ di mana $\mathbf{P} = \mathbf{K} [\mathbf{R} \mid \mathbf{t}]$.

Untuk dua pengamatan piksel $\mathbf{x}_1 = (u_1, v_1)$ dan $\mathbf{x}_2 = (u_2, v_2)$, sistem Direct Linear Transformation membentuk matriks $\mathbf{A}_{4 \times 4}$:
$$\begin{bmatrix} u_1 \mathbf{p}_1^{3T} - \mathbf{p}_1^{1T} \\ v_1 \mathbf{p}_1^{3T} - \mathbf{p}_1^{2T} \\ u_2 \mathbf{p}_2^{3T} - \mathbf{p}_2^{1T} \\ v_2 \mathbf{p}_2^{3T} - \mathbf{p}_2^{2T} \end{bmatrix} \mathbf{X} = \mathbf{0}$$
Solusi $\mathbf{X}$ diperoleh melalui Singular Value Decomposition (SVD): $\mathbf{A} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T$, di mana vektor homogen $\mathbf{X}$ adalah kolom terakhir $\mathbf{V}$ yang meminimalkan ekuasi kuadratik.

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Algoritmik NumPy murni.")

In [ ]:
# 3. Implementasi Triangulasi DLT & Validasi Contoh Analitik
def construct_camera_matrix(f_x, f_y, c_x, c_y, R, t):
    K = np.array([
        [f_x,  0.0, c_x],
        [ 0.0, f_y, c_y],
        [ 0.0,  0.0, 1.0]
    ], dtype=np.float64)
    Rt = np.hstack((R, t.reshape(3, 1)))
    return K @ Rt

def linear_dlt_triangulation(P1, P2, x1, x2):
    u1, v1 = x1[0], x1[1]
    u2, v2 = x2[0], x2[1]
    
    A = np.zeros((4, 4), dtype=np.float64)
    A[0] = u1 * P1[2] - P1[0]
    A[1] = v1 * P1[2] - P1[1]
    A[2] = u2 * P2[2] - P2[0]
    A[3] = v2 * P2[2] - P2[1]
    
    _, _, Vt = np.linalg.svd(A)
    X_hom = Vt[-1]
    return X_hom[:3] / X_hom[3]

def project_point(P, X_3d):
    X_hom = np.append(X_3d, 1.0)
    x_proj_hom = P @ X_hom
    return x_proj_hom[:2] / x_proj_hom[2]

# Kamera 1 di Origin, Kamera 2 bergeser 2.0 meter ke kanan (b=2.0 m)
f = 1000.0
P1 = construct_camera_matrix(f, f, 0.0, 0.0, np.eye(3), np.zeros(3))
P2 = construct_camera_matrix(f, f, 0.0, 0.0, np.eye(3), np.array([-2.0, 0.0, 0.0]))

x1 = np.array([200.0, 100.0])
x2 = np.array([-200.0, 100.0])

X_sol = linear_dlt_triangulation(P1, P2, x1, x2)
print(f"Hasil Triangulasi DLT 3D: X = {X_sol[0]:.4f}m, Y = {X_sol[1]:.4f}m, Z = {X_sol[2]:.4f}m")

# Assert ground truth analitis [1.0, 0.5, 5.0]
np.testing.assert_allclose(X_sol, np.array([1.0, 0.5, 5.0]), atol=1e-5)
print("VALIDASI BERHASIL: Solusi SVD DLT persis sama dengan ground truth analitik!")

In [ ]:
# 4. Evaluasi Eror Reproyeksi & Rekonstruksi Titik Multi-Objek
# Buat cloud titik sintesis 3D kubus (8 titik sudut)
corners_3d = np.array([
    [-0.5, -0.5, 4.0],
    [ 0.5, -0.5, 4.0],
    [ 0.5,  0.5, 4.0],
    [-0.5,  0.5, 4.0],
    [-0.5, -0.5, 5.0],
    [ 0.5, -0.5, 5.0],
    [ 0.5,  0.5, 5.0],
    [-0.5,  0.5, 5.0]
])

reconstructed_cloud = []
rmse_list = []

for X_pt in corners_3d:
    u1_pt = project_point(P1, X_pt)
    u2_pt = project_point(P2, X_pt)
    
    X_est = linear_dlt_triangulation(P1, P2, u1_pt, u2_pt)
    reconstructed_cloud.append(X_est)
    
    err1 = np.linalg.norm(project_point(P1, X_est) - u1_pt)
    err2 = np.linalg.norm(project_point(P2, X_est) - u2_pt)
    rmse_list.append(np.sqrt(0.5 * (err1**2 + err2**2)))

mean_rmse = np.mean(rmse_list)
print(f"Rekonstruksi {len(corners_3d)} Titik 3D Kubus Selesai.")
print(f"Rata-rata Eror Reproyeksi: {mean_rmse:.8f} piksel")
assert mean_rmse < 1e-6, "Eror reproyeksi numerik melampaui toleransi nol ideal!"
print("VERIFIKASI GEOMETRI KUBUS: Lulus deterministik.")

In [ ]:
# 5. Spesifikasi Pipeline PyTorch / Deep Learning (Untuk GPU & Colab)
if HAS_TORCH:
    print("Mode PyTorch / Deep Learning Aktif.")
    P1_t = torch.tensor(P1, dtype=torch.float32)
    P2_t = torch.tensor(P2, dtype=torch.float32)
    print("Camera Tensor P1:", P1_t.shape)
    print("Camera Tensor P2:", P2_t.shape)
else:
    print("CATATAN AKADEMIK UNTUK EKSEKUSI CLOUD / GOOGLE COLAB:")
    print("1. Untuk rekonstruksi fotogrametri nyata, pasang pycolmap atau PyTorch3D:")
    print("   !pip install opencv-python-headless scipy torch torchvision")
    print("2. Pipeline Structure from Motion (SfM) end-to-end:")
    print("   - Ekstraksi fitur: cv2.SIFT_create()")
    print("   - Pencocokan fitur: cv2.FlannBasedMatcher()")
    print("   - Estimasi Matriks Esensial: cv2.findEssentialMat(..., method=cv2.RANSAC)")
    print("   - Pemulihan Pose: cv2.recoverPose(E, points1, points2)")
    print("   - Triangulasi: cv2.triangulatePoints(P1, P2, points1.T, points2.T)")
    print("   - Bundle Adjustment: scipy.optimize.least_squares()")

### 6. Rangkuman & Pembahasan Akademik
1. **Triangulasi Geometri Multi-Pandang**: Menggunakan prinsip DLT dan SVD, titik 3D dapat dipulihkan secara matematis dari korespondensi piksel 2D yang terkalibrasi.
2. **Sensitivitas Kedalaman Terhadap Disparitas**: Eror kedalaman $\Delta Z$ berbanding lurus dengan kuadrat jarak $Z^2$ dan berbanding terbalik dengan baseline $b$.
3. **Langkah Lanjutan**: Untuk data citra nyata dengan gangguan pencahayaan dan distorsi lensa, optimasi Bundle Adjustment non-linier adalah instrumen wajib untuk mengoreksi akumulasi drift rekonstruksi.